In [13]:
from src.utils.spark_session import create_spark_session

from pyspark.sql import functions as F
from pyspark.sql import DataFrame, SparkSession
import logging


logger = logging.getLogger(__name__)

In [9]:
spark = create_spark_session()

In [10]:
df = spark.read.csv("data/raw/accounts", header=True)

In [11]:
df.show()

+-----------+-----------+------------+--------------+----------+----------+--------+---------+-------------+---------+------------------+
| account_id|customer_id|account_type|account_status| open_date|close_date|currency|branch_id|interest_rate|  balance|last_activity_date|
+-----------+-----------+------------+--------------+----------+----------+--------+---------+-------------+---------+------------------+
|ACC00000000| CUST000363| investments|        active|2025-09-16|      null|     GBP|    BR498|    1.2346787|98345.266|        2025-08-20|
|ACC00000001| CUST000209|    checking|        closed|2022-09-17|2025-09-10|     GBP|    BR709|    1.2244802| 75040.76|        2025-09-12|
|ACC00000002| CUST000796|    checking|        active|2021-09-17|      null|     EUR|    BR262|    1.8961781|39435.562|        2025-09-14|
|ACC00000003| CUST000888| investments|        closed|2023-09-17|2024-11-15|     GBP|    BR999|   0.63443124|15524.992|        2025-08-19|
|ACC00000004| CUST000460| investme

In [7]:
df.printSchema()

root
 |-- account_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- account_type: string (nullable = true)
 |-- account_status: string (nullable = true)
 |-- open_date: string (nullable = true)
 |-- close_date: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- branch_id: string (nullable = true)
 |-- interest_rate: string (nullable = true)
 |-- balance: string (nullable = true)
 |-- last_activity_date: string (nullable = true)



In [21]:
def clean_account_data(df: DataFrame) -> DataFrame:
        """
        Clean accounts data by handling missing values and data types

        Args:
            df (DataFrame): Raw accounts data
        
        Returns:
            DataFrame: Cleaned accounts dataframe
        """
        logger.info("Cleaning account data")

        # Convert date strings to dates
        df = df.withColumn("open_date", F.to_date("open_date"))

        df = df.withColumn("close_date", F.to_date("close_date"))

        df = df.withColumn("last_activity_date", F.to_date("last_activity_date"))

        # Convert string type to numeric
        df = df.withColumn("interest_rate", F.col("interest_rate").astype("decimal(3,2)"))

        df = df.withColumn("balance", F.col("balance").astype("decimal(12,2)"))

        # Fill missing values
        df = df.fillna("N/A", ["account_type", "account_status", "currency", "branch_id"])

        return df

        

        

In [22]:
df1 = clean_account_data(df)

In [23]:
df1.printSchema()

root
 |-- account_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- account_type: string (nullable = false)
 |-- account_status: string (nullable = false)
 |-- open_date: date (nullable = true)
 |-- close_date: date (nullable = true)
 |-- currency: string (nullable = false)
 |-- branch_id: string (nullable = false)
 |-- interest_rate: decimal(3,2) (nullable = true)
 |-- balance: decimal(12,2) (nullable = true)
 |-- last_activity_date: date (nullable = true)



In [24]:
df1.show()

+-----------+-----------+------------+--------------+----------+----------+--------+---------+-------------+--------+------------------+
| account_id|customer_id|account_type|account_status| open_date|close_date|currency|branch_id|interest_rate| balance|last_activity_date|
+-----------+-----------+------------+--------------+----------+----------+--------+---------+-------------+--------+------------------+
|ACC00000000| CUST000363| investments|        active|2025-09-16|      null|     GBP|    BR498|         1.23|98345.27|        2025-08-20|
|ACC00000001| CUST000209|    checking|        closed|2022-09-17|2025-09-10|     GBP|    BR709|         1.22|75040.76|        2025-09-12|
|ACC00000002| CUST000796|    checking|        active|2021-09-17|      null|     EUR|    BR262|         1.90|39435.56|        2025-09-14|
|ACC00000003| CUST000888| investments|        closed|2023-09-17|2024-11-15|     GBP|    BR999|         0.63|15524.99|        2025-08-19|
|ACC00000004| CUST000460| investments|   